# SSM extension: DDM × Flexible PMC + RDM × Flexible PMC

Sequential-sampling-model versions of the paper's main behavioural
comparison (`comprehensive_model_comparison.ipynb`). The SSMs add a
WFPT/race likelihood over the joint (choice, RT) data, on top of the
same Bayesian-observer front-end used by the Flexible PMC model.
Following bauer's lesson 8: by adding the RT likelihood, the SSMs
should produce **tighter posteriors** on the shared cognitive
parameters (the noise splines, the priors) without sacrificing
choice fit.

## SSM ↔ PMC analogy

| PMC variant (paper Table 1) | DDM analogue | RDM analogue |
|---|---|---|
| `11_null` (Weber, no TMS) | `ddm_weber_null` | `rdm_weber_null` |
| `11b` (Weber, TMS on memory) | `ddm_weber_memory` | `rdm_weber_memory` |
| `11c` (Weber, TMS on perceptual) | `ddm_weber_perception` | `rdm_weber_perception` |
| `11a` (Weber, TMS on both) | `ddm_weber` | `rdm_weber` |
| `flexible2_null` (Flexible, no TMS) | `ddm_flexible_null` | `rdm_flexible_null` |
| `flexible2b` (Flexible, TMS on perceptual) | `ddm_flexible_perception` | `rdm_flexible_perception` |
| `flexible2a` (Flexible, TMS on memory) | `ddm_flexible_memory` | `rdm_flexible_memory` |
| `flexible2` (Flexible, TMS on both) | `ddm_flexible` | `rdm_flexible` |

Each row uses the **same** 5-spline noise function over magnitude,
the **same** subject-level random effects, the **same** prior beliefs
over risky / safe payoff distributions. The only structural
difference is the likelihood: Bernoulli(choice) for PMC vs.
WFPT/race(choice, RT) for SSM.

Two extra DDM/RDM variants probe an alternative mechanism — caution
shift via the accumulator threshold `a` — that the choice-only PMC
model can't distinguish from noise increase:

| Extra variant | TMS regressor |
|---|---|
| `*_threshold` | threshold `a` only (no noise effect) |
| `*_noise_threshold` | both noise splines and `a` |

In [ ]:
bids_folder = '/data/ds-tmsrisk/'

import os.path as op
import arviz as az
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load traces

In [ ]:
# Model labels mapping the paper's 8 PMC variants (4 Weber + 4 Flexible)
# onto SSM analogues (DDM + RDM). The full 20-trace set is:
#
#   4 PMC reference labels (paper Table 1, choice-only)
#   8 Weber-noise SSM labels  (4 DDM + 4 RDM, paper 11_* analogues)
#   12 Flexible-noise SSM labels (DDM/RDM × 6 — 4 paper analogues + 2 extras)

pmc_labels = ['flexible2_null', 'flexible2b', 'flexible2a', 'flexible2',
              '11_null', '11b', '11c', '11a']

ddm_weber_labels = ['ddm_weber_null', 'ddm_weber_perception',
                    'ddm_weber_memory', 'ddm_weber']
rdm_weber_labels = [l.replace('ddm_', 'rdm_') for l in ddm_weber_labels]

ddm_flex_labels = ['ddm_flexible_null', 'ddm_flexible_perception',
                   'ddm_flexible_memory', 'ddm_flexible',
                   'ddm_flexible_threshold', 'ddm_flexible_noise_threshold']
rdm_flex_labels = [l.replace('ddm_', 'rdm_') for l in ddm_flex_labels]

ddm_labels = ddm_weber_labels + ddm_flex_labels
rdm_labels = rdm_weber_labels + rdm_flex_labels

all_labels = pmc_labels + ddm_labels + rdm_labels

# Pretty names — mirror the analogy in the markdown above
pmc_pretty = {
    'flexible2_null': 'PMC Flexible (null)',
    'flexible2b':     'PMC Flexible (TMS on perceptual)',
    'flexible2a':     'PMC Flexible (TMS on memory)',
    'flexible2':      'PMC Flexible (TMS on both)',
    '11_null':        'PMC Weber (null)',
    '11b':            'PMC Weber (TMS on memory)',
    '11c':            'PMC Weber (TMS on perceptual)',
    '11a':            'PMC Weber (TMS on both)',
}
def _ssm_pretty(prefix, kind):
    return {
        f'{prefix}_null':            f'{kind} Weber (null)',
        f'{prefix}_perception':      f'{kind} Weber (TMS on perceptual)',
        f'{prefix}_memory':          f'{kind} Weber (TMS on memory)',
        f'{prefix}':                 f'{kind} Weber (TMS on both)',
    }
ddm_weber_pretty = _ssm_pretty('ddm_weber', 'DDM')
rdm_weber_pretty = _ssm_pretty('rdm_weber', 'RDM')

ddm_flex_pretty = {
    'ddm_flexible_null':            'DDM Flexible (null)',
    'ddm_flexible_perception':      'DDM Flexible (TMS on perceptual)',
    'ddm_flexible_memory':          'DDM Flexible (TMS on memory)',
    'ddm_flexible':                 'DDM Flexible (TMS on both)',
    'ddm_flexible_threshold':       'DDM Flexible (TMS on threshold)',
    'ddm_flexible_noise_threshold': 'DDM Flexible (TMS on both + threshold)',
}
rdm_flex_pretty = {k.replace('ddm', 'rdm'): v.replace('DDM', 'RDM')
                   for k, v in ddm_flex_pretty.items()}

mapping = {**pmc_pretty,
           **ddm_weber_pretty, **rdm_weber_pretty,
           **ddm_flex_pretty, **rdm_flex_pretty}


In [ ]:
from tqdm.notebook import tqdm
from tms_risk.behavior.fit_model import build_model, get_data
import pymc as pm

def load_trace_with_loglik(label):
    """Load a trace, ensuring its log_likelihood group is populated.

    Recent fits (post the fit_model.py LL-save patch) have log_likelihood
    baked in. Older fits don't — for those we rebuild the model and call
    pm.compute_log_likelihood post-hoc, the trick the paper's Table-1
    notebook uses. Rebuilds for PMC labels work locally; rebuilds for
    ddm_*/rdm_* labels would need hssm — for those we skip with a warning.
    """
    fn = op.join(bids_folder, 'derivatives', 'cogmodels', f'model-{label}_trace.netcdf')
    if not op.exists(fn):
        return None, 'MISSING'
    idata = az.from_netcdf(fn).sel(draw=slice(None, None, 2))
    if 'log_likelihood' in idata.groups():
        return idata, 'LL_BAKED_IN'
    needs_hssm = label.startswith('ddm_') or label.startswith('rdm_')
    try:
        df = get_data(model_label=label, bids_folder=bids_folder)
        model = build_model(label, df)
        model.build_estimation_model()
        with model.estimation_model:
            pm.compute_log_likelihood(idata)
        return idata, 'LL_RECOMPUTED'
    except Exception as e:
        msg = 'SKIP (needs hssm)' if needs_hssm else f'SKIP ({type(e).__name__})'
        return None, msg

idatas = {}
for label in tqdm(all_labels):
    idata, status = load_trace_with_loglik(label)
    if idata is None:
        print(f'  {status}: {label}')
        continue
    idatas[label] = idata

print(f'\nLoaded {len(idatas)} / {len(all_labels)} traces with log_likelihood')
labeled = {mapping[k]: v for k, v in idatas.items()}


## ELPD comparison (Table-1 analogue)

Same structure as `comprehensive_model_comparison.ipynb`: pool every
model into one `az.compare(..., ic='loo')` call. The within-family
rankings below partition this for legibility.

In [ ]:
# az.compare can't pool PMC (choice-only) and SSM (choice+RT) likelihoods —
# different observation counts. The per-family rankings below are the
# meaningful comparisons; cross-family interpretation is qualitative.
print('--- Per-family LOO ranks below ---')


In [ ]:
# (joint plot_compare removed — see per-family rankings below)


In [ ]:
def family(labels):
    return {mapping[k]: idatas[k] for k in labels if k in idatas}

for name, family_labels in [
        ('Paper PMC family',  pmc_labels),
        ('DDM Weber',         ddm_weber_labels),
        ('RDM Weber',         rdm_weber_labels),
        ('DDM Flexible',      ddm_flex_labels),
        ('RDM Flexible',      rdm_flex_labels)]:
    fam = family(family_labels)
    if len(fam) < 2:
        print(f'--- {name}: skipped (only {len(fam)} traces) ---')
        continue
    print(f'\n--- {name} ---')
    display(az.compare(fam, ic='loo'))

# Per-family plot_compare panels
import matplotlib.pyplot as plt
_panels = [
    ('Paper PMC',      pmc_labels),
    ('DDM Weber',      ddm_weber_labels),
    ('RDM Weber',      rdm_weber_labels),
    ('DDM Flexible',   ddm_flex_labels),
    ('RDM Flexible',   rdm_flex_labels),
]
fig, axes = plt.subplots(1, len(_panels), figsize=(4*len(_panels), 3.5), sharey=False)
for ax, (name, fam_labels) in zip(axes, _panels):
    fam = {mapping[k]: idatas[k] for k in fam_labels if k in idatas}
    if len(fam) < 2:
        ax.set_title(f'{name}: too few traces')
        ax.axis('off')
        continue
    az.plot_compare(az.compare(fam, ic='loo'), ax=ax, plot_ic_diff=True, plot_standard_error=True)
    ax.set_title(name)
plt.tight_layout()


## Posterior tightening: HDI widths vs PMC (lesson-8 style)

Per bauer's tutorial lesson 8: if the SSM is well-specified, joint
(choice, RT) likelihood should produce **tighter** posteriors on the
shared cognitive parameters than the choice-only PMC.

Compare the 'TMS on both' variant of each family: PMC `flexible2` vs
DDM `ddm_flexible` vs RDM `rdm_flexible`.

In [ ]:
def hdi_widths(idata, prefix):
    """94% HDI widths for every group-mu variable whose name starts with `prefix`."""
    rows = []
    post = idata.posterior
    for var in post.data_vars:
        if not var.startswith(prefix) or not var.endswith('_mu'):
            continue
        hdi = az.hdi(post[var], hdi_prob=0.94)[var].values
        flat = hdi.reshape(-1, 2)
        widths = flat[:, 1] - flat[:, 0]
        for i, w in enumerate(widths):
            rows.append({'parameter': var, 'index': i, 'width': float(w)})
    return pd.DataFrame(rows)

# Look at the noise splines specifically — they're the shared front-end.
pmc_w = hdi_widths(idatas['flexible2'], 'memory_noise_sd_spline').assign(model='PMC')
frames = [pmc_w]
for k, name in [('ddm_flexible', 'DDM'), ('rdm_flexible', 'RDM')]:
    if k in idatas:
        frames.append(hdi_widths(idatas[k], 'memory_noise_sd_spline').assign(model=name))
widths = pd.concat(frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(6, 4))
sns.stripplot(data=widths, x='parameter', y='width', hue='model', dodge=True, ax=ax, alpha=0.6)
ax.set_title('Group-μ 94% HDI width — memory noise splines')
ax.set_ylabel('HDI width (lower = tighter)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

## What to look for

1. **Within each family, the bare (`*_flexible`) variant is close to or beats `*_null`.** Confirms the noise effect is detectable jointly with RT data, not just from choices.
2. **`*_threshold` ranks below `*_perception` / `*_memory` / bare.** Rules out the simpler caution-shift account: cTBS doesn't just shift accumulator boundaries, it inflates perceptual noise.
3. **`*_noise_threshold` doesn't decisively beat `*_flexible`.** Tells us we don't need to add a caution-shift component to the paper's claim.
4. **The HDI-width plot shows DDM/RDM bars below the PMC bar.** That's the lesson-8 promise: RT information tightens the noise-spline posteriors.